In [ ]:
import fitz  # PyMuPDF
import numpy as np
from PIL import Image

In [ ]:
def pdf_page_to_numpy_bitmap(pdf_path, page_number=0, zoom=2.0):
    """
    Converts a specified page of a PDF document into a NumPy array representing a bitmap.

    Args:
        pdf_path (str): The path to the PDF file.
        page_number (int): The 0-indexed number of the page to convert.
        zoom (float): The zoom factor for rendering the page (e.g., 2.0 for 200% resolution).

    Returns:
        numpy.ndarray: A 3D NumPy array (height, width, channels) representing the bitmap,
                        or None if the PDF or page cannot be opened.
    """
    try:
        doc = fitz.open(pdf_path)
        page = doc.load_page(page_number)

        # Create a transformation matrix for zooming
        mat = fitz.Matrix(zoom, zoom)

        # Render the page to a Pixmap
        pix = page.get_pixmap(matrix=mat)

        # Convert the Pixmap samples to a NumPy array
        # The .samples attribute contains the raw pixel data
        # Reshape it to (height, width, channels)
        # pix.n is the number of color channels (e.g., 3 for RGB, 4 for RGBA)
        bitmap_array = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.h, pix.w, pix.n)

        # If the pixmap is RGBA and you need RGB, you can discard the alpha channel:
        if pix.n == 4:
            bitmap_array = bitmap_array[:, :, :3] # Keep only RGB channels

        doc.close()
        return bitmap_array
    except Exception as e:
        print(f"Error converting PDF page to bitmap: {e}")
        return None

mat = pdf_page_to_numpy_bitmap(
    "/home/tony/workspace/projects/Toluene_resistence_evolution/data/lipids/27Aug2025/Size_Total_27Aug2025.pdf",
    0, 10,
)
mat.shape

In [ ]:
H, W, _ = mat.shape
legend = mat[H-(H//5):]
legend.shape

In [ ]:
Image.fromarray(legend).save("./cache/legend.bmp")

In [ ]:
_f = (legend != 255).sum(axis=2).astype(bool)
_f = _f&(legend != 0).sum(axis=2).astype(bool)

m = legend[_f]
_f_rg = m[:, 0] == m[:, 1]
_f_gb = m[:, 2] == m[:, 1]
unique_colors = m[~(_f_rg&_f_gb)]
unique_colors

In [ ]:
from local.figures.template import BaseFigure, ApplyTemplate, go
from local.figures.colors import Color, COLORS, Palettes
from local.caching import save, load, save_exists

In [ ]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=32, init='k-means++', random_state=42, n_init='auto')
clust = kmeans.fit_transform(unique_colors)
labels = kmeans.labels_
labels

In [ ]:
from umap import UMAP

NAME = "colors_emb"
if save_exists(NAME):
    emb = load(NAME)
else:
    model = UMAP()
    emb = model.fit_transform(unique_colors, y=labels)
    save(NAME, emb)
emb.shape

In [ ]:
fig = BaseFigure()

_black = Color.Hex('212121')
fig.add_trace(
    go.Scattergl(
        x = emb[:, 0],
        y = emb[:, 1],
        mode="markers",
        marker=dict(color=labels),
    ),
)
fig = ApplyTemplate(
    fig,
)
fig.show()

In [ ]:
lmean = legend.mean(axis=2)
legend_bw = legend.copy()
legend_bw[:, :, 0] = lmean
legend_bw[:, :, 1] = lmean
legend_bw[:, :, 2] = lmean

Image.fromarray(legend_bw).save("./cache/legend_bw.bmp")

In [ ]:
labels = np.array(labels)
colors = {}
for i in range(32):
    _f = labels == i
    if i in {2, 14}: continue
    if _f.sum() < 300: continue
    c = unique_colors[_f].mean(axis=0)
    colors[i] = c
    dist = legend-c
    _hits = np.sqrt(np.square(dist).sum(axis=2)) < 5
    _legend = legend_bw.copy()
    _legend[_hits] = [255, 0, 0]
    _legend[~_hits] = (_legend[~_hits]*2).clip(0, 255)
    Image.fromarray(_legend).save(f"./cache/{i}.bmp")

In [ ]:
mat.shape, mat.shape[0]//5, mat.shape[1]//5

In [ ]:
h, w = 1584, 1224
l, t = w*2, int(np.round(h*3.3))
graph = mat[t:t+int(np.round(h*0.7)), l:l+int(np.round(w*1.5))]
Image.fromarray(graph).save(f"./cache/graph.bmp")

In [ ]:
graph.shape

In [ ]:
xx = list(range(graph.shape[1]))
xx = np.repeat(np.array(xx)[:, np.newaxis], graph.shape[0], axis=1).T
yy = list(range(graph.shape[0]))
yy = np.repeat(np.array(yy)[:, np.newaxis], graph.shape[1], axis=1)
xx.shape, yy.shape

In [ ]:
g = graph.copy()
x10 = 365
x100 = 1055
x1k = 1753
y0 = 871
y2 = 711
y4 = 554
y6 = 404
y8 = 246
y10 = 88
g[y0] = [255, 0, 0]
Image.fromarray(g).save(f"./cache/graphxy.bmp")

y_grid = [y0, y2, y4, y6, y8, y10]
x_grid = [x10, x100, x1k]

f_grid = np.zeros(shape=(graph.shape[0], graph.shape[1])).astype(bool)
th = 5
for y in y_grid:
    _f = np.abs(yy-y) < th
    f_grid |= _f
for x in x_grid:
    _f = np.abs(xx-x) < th
    f_grid |= _f

y_mask = yy>859
# y_mask = yy>880

g = graph.copy()
g[f_grid|y_mask] = [255, 255, 255]
Image.fromarray(g).save(f"./cache/graph_grid.bmp")

In [ ]:
colors[-1] = np.array([0, 0, 0])

In [ ]:
from scipy.optimize import curve_fit, minimize_scalar

def pix2size(x, x0, a):
    return a*(x-x0)

def fit_size(x, y):
    initial_guess = [0, 1]
    params, covariance = curve_fit(pix2size, x, y, p0=initial_guess, maxfev=10000)
    return params

def pix2int(x, x0, a):
    return a*(x-x0)

def fit_int(x, y):
    initial_guess = [0, 1]
    params, covariance = curve_fit(pix2int, x, y, p0=initial_guess, maxfev=10000)
    return params

psize = fit_size(x_grid, [1, 2, 3])
x_size = np.pow(10, pix2size(xx[0], *psize))
pint = fit_int(y_grid, [0, 2, 4, 6, 8, 10])
y_int = pix2int(yy[:, 0], *pint)

In [ ]:
k2name = {
    13:     ("Clone1", "1"),
    1:      ("Clone1", "2"),
    9:      ("Clone1", "3"),
    3:      ("Clone2", "1"),
    5:      ("Clone2", "2"),
    6:      ("Clone2", "3"),
    11:     ("PCC1", "1"),
    10:     ("PCC1", "2"),
    8:      ("PCC1", "3"),
    -1:     ("EPI300", "1"),
    4:      ("EPI300", "2"),
    0:      ("EPI300", "3"),
}

In [ ]:
data = []
filter_masks = {}
for k in colors:
    _c = np.round(colors[k]).astype(int)
    _f = ((graph == _c).sum(axis=2) == 3)
    if k == -1:
        _f = _f & ~(f_grid|y_mask)
        _c = [255, 255, 255]
    if k == 9:
        _f[:, 810:820] = False
        _f[:, 1387:1396] = False
    if k == -1:
        _f[:, 1185:] = filter_masks[0][:, 1185:]
    filter_masks[k] = _f
    g = np.zeros_like(graph)
    g[_f] = _c

    _f_int = _f.astype(int)
    edges = _f_int[1:]-_f_int[:-1]
    _y1 = np.argmax(edges, axis=0)
    _y2 = np.argmin(edges, axis=0)
    _y = (_y1+_y2)/2
    _y[_y==0] = np.nan
    # _y = y0-_y
    _ynan = ~np.isnan(_y).astype(int)
    first = _ynan.argmax()
    _ynan = ~np.isnan(_y[::-1]).astype(int)
    last = len(_ynan)-_ynan.argmax()
    mask = np.isnan(_y)
    _y[mask] = np.interp(np.flatnonzero(mask), np.flatnonzero(~mask), _y[~mask])
    _y[:first] = np.nan
    _y[last:] = np.nan
    _y = _y.clip(0, np.inf)
    _y_int = pix2int(_y, *pint)
    name, rep = k2name[k]
    data.append((name, rep, _y_int))

In [ ]:
_cond = {}
for name, rep, y in data:
    _cond[name] = _cond.get(name, [])+[(rep, y)]

_order = [
    "Clone1",
    "Clone2",
    "PCC1",
    "EPI300",
]
by_cond = {k:_cond[k] for k in _order}
by_cond

In [ ]:
by_cond.keys()

In [ ]:
fig = BaseFigure(
    shape=(4, 1),
    x_title="Intensity (%)",
)
for ann in fig.layout["annotations"]:
    if ann["x"] == 0.5:
        ann["y"] -= 0.1
        ann["font"]["size"]=20

_colors = {
    "Clone1": Palettes.PLOTLY[0],
    "Clone2": Palettes.PLOTLY[1],
    "PCC1": Color.Hex("555555"),
    "EPI300": Color.Hex("999999"),
}
col = {
    "Clone1":   1,
    "Clone2":   2,
    "PCC1":     3,
    "EPI300":   4,
}

for name, reps in by_cond.items():
    for rep, y in reps:
        fig.add_trace(
            go.Scatter(
                name=name,
                y = x_size,
                x = y.clip(0, np.inf),
                mode="lines",
                fill="tozeroy",
                fillcolor=_colors[name].Fade(0.2).color_value,
                line=dict(color=_colors[name].color_value, width=1),
                showlegend=False,
            ),
            row=1, col=col[name],
        )
    fig.add_trace(
        go.Scatter(
            name=name,
            y = [110, 110, None, 25, 25],
            x = [-99, 99, None, -99, 99],
            mode="lines",
            line=dict(color=COLORS.BLACK, width=1),
            showlegend=False,
        ),
        row=1, col=col[name],
    )

invis = dict(showticklabels=False, linecolor=COLORS.TRANSPARENT, ticks=None)
show = dict(showticklabels=True, linecolor=COLORS.BLACK, ticks="outside")
fig = ApplyTemplate(
    fig,
    default_yaxis=invis|dict(range=[1, np.log10(500)], type="log"),
    # default_yaxis=invis|dict(dtick=100, range=[0, 500]),
    default_xaxis=dict(dtick=8, range=[-2, 12]),
    axis = {
        # "1 1 x": dict(type="log"),
        # "1 1 y": show|dict(dtick=100, range=[0, 300]),
        "1 1 y": show|dict(title="particle size (nm)"),
    } | {
        f"{x+1} 1 x": dict(title=dict(text=f'<span style="color:{_colors[t].AsHex()}">{t}</span>'))
        for x, t in enumerate(by_cond)
    },
    layout=dict(
        width=800, height=500,
        margin=dict(t=20, b=100),
    ),
)
fig.write_image("./cache/DLS_raw.png")
fig.show()